# FlowPolicy — Franka Kitchen di Google Colab

**Sebelum mulai:** Runtime → *Change runtime type* → **GPU** (T4/L4/A100).

**Catatan:** Instalasi pertama panjang (~30–60 menit) karena MuJoCo, `mujoco-py`, Gym 0.21, D4RL, dan PyTorch3D. Disarankan **unggah file zarr** dataset dari laptop (Google Drive) daripada generate ulang di Colab.

Struktur yang diharapkan setelah clone:
`/content/FlowPolicy/` berisi `FlowPolicy/`, `third_party/`, `scripts/`, `colab/`.

In [ ]:
# Cek GPU
!nvidia-smi

In [ ]:
%%bash
set -e
apt-get update -qq
apt-get install -y -qq \
  build-essential patchelf wget git ffmpeg \
  libgl1-mesa-dev libgl1-mesa-glx libglew-dev libosmesa6-dev libx11-dev

In [ ]:
%%bash
set -e
cd /content
if [ ! -d "$HOME/miniconda" ]; then
  wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O miniconda.sh
  bash miniconda.sh -b -p "$HOME/miniconda"
fi
export PATH="$HOME/miniconda/bin:$PATH"
source "$HOME/miniconda/etc/profile.d/conda.sh"
if ! conda env list | grep -qE '^flowpolicy\s'; then
  conda create -n flowpolicy python=3.8 -y
fi

## Clone repositori

Ganti URL di bawah dengan repo Anda (GitHub/GitLab). Repo **pribadi**: gunakan token atau unggah zip lewat panel Files Colab lalu `!unzip FlowPolicy-main.zip -d /content && mv /content/FlowPolicy-main /content/FlowPolicy`.

In [ ]:
import os, shutil, subprocess
os.chdir('/content')
REPO_URL = 'https://github.com/USERNAME/FlowPolicy.git'  # <-- GANTI URL repo Anda
if os.path.isdir('FlowPolicy'):
    shutil.rmtree('FlowPolicy')
subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, 'FlowPolicy'], check=True)

In [ ]:
%%bash
set -e
mkdir -p "$HOME/.mujoco"
cd "$HOME/.mujoco"
if [ ! -d mujoco210 ]; then
  wget -q https://github.com/deepmind/mujoco/releases/download/2.1.0/mujoco210-linux-x86_64.tar.gz -O mj210.tar.gz
  tar -xzf mj210.tar.gz
  rm -f mj210.tar.gz
fi
ls -la "$HOME/.mujoco/mujoco210/bin" | head

In [ ]:
%%bash
set -e
source "$HOME/miniconda/etc/profile.d/conda.sh"
conda activate flowpolicy
export REPO_DIR=/content/FlowPolicy
export MUJOCO_GL=osmesa
export LD_LIBRARY_PATH="${LD_LIBRARY_PATH}:${HOME}/.mujoco/mujoco210/bin:/usr/lib/nvidia"
chmod +x "$REPO_DIR/colab/colab_install_flowpolicy.sh"
bash "$REPO_DIR/colab/colab_install_flowpolicy.sh"

## Dataset (zarr)

1. Unggah `franka_kitchen_expert.zarr` ke **Google Drive** (folder zip atau folder zarr).
2. Mount Drive di sel berikut, lalu salin ke `/content/FlowPolicy/data/`.

Jika zarr berupa folder, pastikan path di `task.dataset.zarr_path` mengarah ke folder tersebut (bukan file).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%%bash
mkdir -p /content/FlowPolicy/data
# Sesuaikan path Drive Anda (contoh):
# cp -r "/content/drive/MyDrive/FlowPolicy_data/franka_kitchen_expert.zarr" /content/FlowPolicy/data/
echo "Edit sel ini: salin zarr dari Drive ke /content/FlowPolicy/data/"

## (Opsional) Weights & Biases

Untuk offline: `logging.mode=offline` di perintah train. Untuk online, jalankan `wandb login` di terminal.

In [ ]:
%%bash
source "$HOME/miniconda/etc/profile.d/conda.sh"
conda activate flowpolicy
# wandb login  # hapus komentar jika perlu

## Training

Preset `flowpolicy_franka_kitchen_8gb` cocok untuk GPU Colab (VRAM terbatas). Sesuaikan `zarr_path`.

In [ ]:
%%bash
set -e
source "$HOME/miniconda/etc/profile.d/conda.sh"
conda activate flowpolicy
export MUJOCO_GL=osmesa
export LD_LIBRARY_PATH="${LD_LIBRARY_PATH}:${HOME}/.mujoco/mujoco210/bin:/usr/lib/nvidia"
export D4RL_SUPPRESS_IMPORT_ERROR=1
cd /content/FlowPolicy/FlowPolicy
python train.py --config-name=flowpolicy_franka_kitchen_8gb \
  task.dataset.zarr_path=/content/FlowPolicy/data/franka_kitchen_expert.zarr \
  training.device=cuda:0 \
  logging.mode=offline